# Hürlimann credible loss ratio reserving: Tables 7.1, 7.2, 7.6 and 7.7

Verifies `cl.HurlimannCLR` against the numerical examples of Hürlimann, *Credible Loss Ratio Claims Reserves: The Benktander, Neuhaus and Mack Methods Revisited*, Astin Bulletin 39(1), 81-99 ([paper](https://www.casact.org/sites/default/files/2021-03/7_Hurlimann.pdf)), for [issue #1502](https://github.com/casact/chainladder-python/issues/1502).

The paper gives two examples, each as a paid triangle and a table of the method's parameters:

| Example | Triangle | Parameters |
| --- | --- | --- |
| Mack (1997), Table 3.1.5.1 | Table 7.1 | Table 7.2 |
| A slightly modified real life project | Table 7.6 | Table 7.7 |

The parameters are the exposure $V$, the loss ratio pattern $m$, the payout and reserve factors $p$ and $q$, and the optimal weight $Z^*=p/(p+t)$ with $t=\sqrt p$. Each is computed by the estimator and asserted against the printed values.

In [1]:
import chainladder as cl
import numpy as np
import pandas as pd


# The paper prints whole numbers or five decimals; tolerance for comparing against its tables.
def close(actual, printed, atol):
    assert np.allclose(actual, printed, atol=atol), (np.round(actual, 5), printed)

In [2]:
def paid_triangle(rows):
    """The paper's incremental triangle, with periods 1 to 6 labelled 2001 to 2006."""
    long = pd.DataFrame(
        [
            (2001 + i, 2001 + i + k, v)
            for i, r in enumerate(rows)
            for k, v in enumerate(r)
        ],
        columns=["origin", "valuation", "paid"],
    )
    return cl.Triangle(
        long, origin="origin", valuation="valuation", columns="paid", cumulative=False
    )


def as_table(incremental):
    table = incremental.to_frame(origin_as_datetime=False)
    table.columns = range(1, 7)
    table.index = range(1, 7)
    return table


def printed_triangle(rows):
    printed = np.full((6, 6), np.nan)
    for i, r in enumerate(rows):
        printed[i, : len(r)] = r
    return printed


def exposure(V):
    """The exposures V_i as a Triangle, to pass as sample_weight."""
    return cl.Triangle(
        pd.DataFrame({"origin": range(2001, 2007), "valuation": 2006, "premium": V}),
        origin="origin",
        valuation="valuation",
        columns="premium",
        cumulative=True,
    )


def parameters(model, V):
    """The columns of Tables 7.2 and 7.7."""

    def values(tri):
        return tri.to_frame().values.ravel()

    p = values(model.p_)
    return pd.DataFrame(
        {
            "V": V,
            # The paper's "m" column lists m_k by development period k, in the same rows as the origins.
            "m": values(model.m_),
            "p": p,
            "q": 1 - p,
            "t": np.sqrt(p),  # the optimal method has f = 1 and t = sqrt(p)
            "Z*": values(model.credibility_),
        },
        index=range(1, 7),
    )

## Table 7.1: the paid triangle (Mack 1997)

The paper's table is incremental, with periods numbered 1 to 6. Here they are labelled 2001 to 2006, with annual development.

In [3]:
rows_7_1 = [
    [4370, 1923, 3999, 2168, 1200, 647],
    [2701, 2590, 1871, 1783, 393],
    [4483, 2246, 3345, 1068],
    [3254, 2550, 2547],
    [8010, 4108],
    [5582],
]
incremental_7_1 = paid_triangle(rows_7_1)
table_7_1 = as_table(incremental_7_1)
table_7_1

,1,2,3,4,5,6
1,4370.0,1923.0,3999.0,2168.0,1200.0,647.0
2,2701.0,2590.0,1871.0,1783.0,393.0,NaN
3,4483.0,2246.0,3345.0,1068.0,NaN,NaN
4,3254.0,2550.0,2547.0,NaN,NaN,NaN
5,8010.0,4108.0,NaN,NaN,NaN,NaN
6,5582.0,NaN,NaN,NaN,NaN,NaN


In [4]:
# Printed values, Table 7.1.
assert np.array_equal(table_7_1.values, printed_triangle(rows_7_1), equal_nan=True)

## Table 7.2: parameters of the credible loss ratio method

The exposures $V_i$ are given in the paper. They are passed as a Triangle through `sample_weight`, as for `cl.Benktander` and `cl.CapeCod`. The estimator takes the incremental or cumulative triangle.

In [5]:
V_7_2 = [13_085, 14_258, 16_114, 15_142, 16_905, 20_224]
model_7_2 = cl.HurlimannCLR(method="optimal").fit(
    incremental_7_1, sample_weight=exposure(V_7_2)
)
table_7_2 = parameters(model_7_2, V_7_2)
table_7_2.round(5)

,V,m,p,q,t,Z*
1,13085,0.29667,1.00000,0.00000,1.00000,0.50000
2,14258,0.17770,0.94496,0.05504,0.97209,0.49292
3,16114,0.20072,0.88010,0.11990,0.93814,0.48404
4,15142,0.11549,0.75153,0.24847,0.86691,0.46435
5,16905,0.05826,0.52808,0.47192,0.72669,0.42086
6,20224,0.04945,0.33026,0.66974,0.57469,0.36495


In [6]:
# Printed values, Table 7.2.
close(table_7_2["V"], V_7_2, 0)
close(table_7_2["m"], [0.29667, 0.17770, 0.20072, 0.11549, 0.05826, 0.04945], 5e-6)
close(table_7_2["p"], [1, 0.94496, 0.88010, 0.75153, 0.52808, 0.33026], 5e-6)
close(table_7_2["q"], [0, 0.05504, 0.11990, 0.24847, 0.47192, 0.66974], 5e-6)
close(table_7_2["t"], [1, 0.97209, 0.93814, 0.86691, 0.72669, 0.57469], 5e-6)
close(table_7_2["Z*"], [0.5, 0.49292, 0.48404, 0.46435, 0.42086, 0.36495], 5e-6)

Tables 7.1 and 7.2 reproduce in full, to the paper's rounding.

## Tables 7.3 and 7.4: reserves and ultimates by method

The five methods differ only in the credibility weight $Z$ on the individual reserve, which is the estimator's `method`. Each is fit on the same triangle and exposure. The "all periods" row of Table 7.3 is the total over origins 2 to 6, since origin 1 is fully paid out ($q_1=0$); Table 7.4's includes origin 1.

In [7]:
methods = ["collective", "individual", "neuhaus", "benktander", "optimal"]
fits_7_2 = {
    method: cl.HurlimannCLR(method=method).fit(
        incremental_7_1, sample_weight=exposure(V_7_2)
    )
    for method in methods
}

reserves = pd.DataFrame(
    {method: fit.ibnr_.to_frame().values.ravel() for method, fit in fits_7_2.items()},
    index=range(1, 7),
).fillna(0)  # ibnr_ is NaN where the origin is fully developed
ultimates = pd.DataFrame(
    {
        method: fit.ultimate_.to_frame().values.ravel()
        for method, fit in fits_7_2.items()
    },
    index=range(1, 7),
)

table_7_3 = pd.concat([
    reserves.iloc[1:].sum().to_frame("all periods").T,
    reserves.iloc[1:],
])
table_7_4 = pd.concat([ultimates.sum().to_frame("all periods").T, ultimates])
table_7_3.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,25154.0,26972.0,25913.0,25999.0,25914.0
2,705.0,544.0,568.0,553.0,626.0
3,1736.0,1518.0,1564.0,1544.0,1630.0
4,3380.0,2761.0,2962.0,2915.0,3092.0
5,7166.0,10829.0,8904.0,9101.0,8708.0
6,12167.0,11320.0,11916.0,11887.0,11858.0


In [8]:
# Printed values, Table 7.3 (all periods, then origins 2 to 6).
printed_7_3 = {
    "collective": [25154, 705, 1736, 3380, 7166, 12167],
    "individual": [26972, 544, 1518, 2761, 10829, 11320],
    "neuhaus": [25913, 568, 1564, 2962, 8904, 11916],
    "benktander": [25999, 553, 1544, 2915, 9101, 11887],
    "optimal": [25914, 626, 1630, 3092, 8708, 11858],
}
for method, printed in printed_7_3.items():
    close(table_7_3[method].values, printed, 1)

Table 7.3 reproduces in full, to the paper's rounding.

In [9]:
table_7_4.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,85992.0,87810.0,86751.0,86837.0,86752.0
1,14307.0,14307.0,14307.0,14307.0,14307.0
2,10043.0,9882.0,9906.0,9891.0,9964.0
3,12878.0,12660.0,12706.0,12686.0,12772.0
4,11731.0,11112.0,11313.0,11266.0,11443.0
5,19284.0,22947.0,21022.0,21219.0,20826.0
6,17749.0,16902.0,17498.0,17469.0,17440.0


In [10]:
# Printed values, Table 7.4 (all periods, then origins 1 to 6), as corrected by the CAS Exam 7
# Syllabus Committee's errata of January 2020. The paper's own printed "collective" and "optimal"
# columns were wrong; the other three columns are unchanged.
printed_7_4 = {
    "collective": [85992, 14307, 10043, 12878, 11731, 19284, 17749],
    "individual": [87810, 14307, 9882, 12660, 11112, 22947, 16902],
    "neuhaus": [86751, 14307, 9906, 12706, 11313, 21022, 17498],
    "benktander": [86837, 14307, 9891, 12686, 11266, 21219, 17469],
    "optimal": [86752, 14307, 9964, 12772, 11443, 20826, 17440],
}
for method, printed in printed_7_4.items():
    close(table_7_4[method].values, printed, 1)

Table 7.4 reproduces in full, to the paper's rounding, **once the errata is applied**. As printed in the paper, its "collective" column is in fact the optimal method's ultimates, and its "optimal" column matches no method (for origin 5, 20,364 implies a reserve of 8,246 against 8,708 in Table 7.3). The errata corrects both.

## Table 7.5: mean squared errors

Table 7.5 reports each method's mean squared error as a ratio to that of the optimal method, which is minimal. It uses the minimum variance estimator of Section 6 with $f_i=1$ and $t_i=\sqrt{p_i}$, for every method. The estimator's `mse_` is the mean squared error of eq. 4.11 in units of the factor $E[a_i^2(U_i)]$, which cancels in a ratio and so is not needed. Origin 1 is omitted, being fully paid out.

In [11]:
mse = pd.DataFrame(
    {method: fit.mse_.to_frame().values.ravel() for method, fit in fits_7_2.items()},
    index=range(1, 7),
).iloc[1:]
table_7_5 = mse.div(mse["optimal"], axis=0)
table_7_5.round(6)

,collective,individual,neuhaus,benktander,optimal
2,1.027133,1.028713,1.014146,1.022818,1.0
3,1.058037,1.065943,1.023277,1.038856,1.0
4,1.115379,1.153525,1.023764,1.044128,1.0
5,1.198610,1.376096,1.003211,1.012892,1.0
6,1.244422,1.740080,1.008555,1.002208,1.0


In [12]:
# Printed values, Table 7.5, as corrected by the CAS Exam 7 Syllabus Committee's errata of
# January 2020. The paper's Neuhaus column, and some last digits of the others, were wrong.
printed_7_5 = {
    "collective": [1.027133, 1.058037, 1.115379, 1.198610, 1.244422],
    "individual": [1.028713, 1.065943, 1.153525, 1.376096, 1.740080],
    "neuhaus": [1.014146, 1.023277, 1.023764, 1.003211, 1.008555],
    "benktander": [1.022818, 1.038856, 1.044128, 1.012892, 1.002208],
    "optimal": [1, 1, 1, 1, 1],
}
for method, printed in printed_7_5.items():
    close(table_7_5[method].values, printed, 5e-6)

Table 7.5 reproduces in full, to the paper's rounding, **once the errata is applied**. The paper as printed has a different Neuhaus column (1.014146, 1.003002, 1.002692, 1.120972, 1.409648), of which only origin 2 is right.

## Table 7.6: the paid triangle (real life project)

In [13]:
rows_7_6 = [
    [3789045, 2860826, 506651, 151996, 65141, 24203],
    [3582774, 2687080, 1250163, 535784, 880143],
    [4221853, 3166390, 2249388, 207853],
    [4074429, 2949557, 1162885],
    [1227618, 3906617],
    [6839930],
]
incremental_7_6 = paid_triangle(rows_7_6)
table_7_6 = as_table(incremental_7_6)
table_7_6

,1,2,3,4,5,6
1,3789045.0,2860826.0,506651.0,151996.0,65141.0,24203.0
2,3582774.0,2687080.0,1250163.0,535784.0,880143.0,NaN
3,4221853.0,3166390.0,2249388.0,207853.0,NaN,NaN
4,4074429.0,2949557.0,1162885.0,NaN,NaN,NaN
5,1227618.0,3906617.0,NaN,NaN,NaN,NaN
6,6839930.0,NaN,NaN,NaN,NaN,NaN


In [14]:
# Printed values, Table 7.6.
assert np.array_equal(table_7_6.values, printed_triangle(rows_7_6), equal_nan=True)

## Table 7.7: parameters of the credible loss ratio method

In [15]:
V_7_7 = [8_000_000, 9_000_000, 10_000_000, 10_000_000, 10_000_000, 12_000_000]
model_7_7 = cl.HurlimannCLR(method="optimal").fit(
    incremental_7_6, sample_weight=exposure(V_7_7)
)
table_7_7 = parameters(model_7_7, V_7_7)
table_7_7.round(5)

,V,m,p,q,t,Z*
1,8000000,0.40230,1.00000,0.00000,1.00000,0.50000
2,9000000,0.33129,0.99687,0.00313,0.99843,0.49961
3,10000000,0.13971,0.93925,0.06075,0.96915,0.49217
4,10000000,0.03317,0.90488,0.09512,0.95125,0.48751
5,10000000,0.05560,0.76012,0.23988,0.87185,0.46577
6,12000000,0.00303,0.41685,0.58315,0.64564,0.39233


In [16]:
# Printed values, Table 7.7. m and q are compared to one unit in the last printed digit,
# the others to half a unit; see the note below.
close(
    table_7_7["V"],
    [8_000_000, 9_000_000, 10_000_000, 10_000_000, 10_000_000, 12_000_000],
    0,
)
close(table_7_7["m"], [0.40230, 0.33129, 0.13971, 0.03317, 0.05561, 0.00303], 1e-5)
close(table_7_7["p"], [1, 0.99687, 0.93925, 0.90488, 0.76012, 0.41685], 5e-6)
close(table_7_7["q"], [0, 0.00314, 0.06075, 0.09512, 0.23988, 0.58315], 1e-5)
close(table_7_7["t"], [1, 0.99843, 0.96915, 0.95125, 0.87185, 0.64564], 5e-6)
close(table_7_7["Z*"], [0.5, 0.49961, 0.49217, 0.48751, 0.46577, 0.39233], 5e-6)

Tables 7.6 and 7.7 reproduce to the paper's rounding, with two entries one unit out in the last printed digit:

- $m_5$ is 0.055605 unrounded, so it rounds to 0.05560, where the paper prints 0.05561.
- $q_2$ is 0.0031348 unrounded, so it rounds to 0.00313, where the paper prints 0.00314. That is also not $1-p_2=1-0.99687$ from the paper's own table, so this looks like a rounding quirk in the printed table rather than a difference in method.

Every other entry is within half a unit of the printed value.

## Tables 7.8 and 7.9: reserves and ultimates by method

As for Tables 7.3 and 7.4, with the second example. The "all periods" row of Table 7.8 is the total over origins 2 to 6, and Table 7.9's includes origin 1.

In [17]:
fits_7_7 = {
    method: cl.HurlimannCLR(method=method).fit(
        incremental_7_6, sample_weight=exposure(V_7_7)
    )
    for method in methods
}

reserves = pd.DataFrame(
    {method: fit.ibnr_.to_frame().values.ravel() for method, fit in fits_7_7.items()},
    index=range(1, 7),
).fillna(0)  # ibnr_ is NaN where the origin is fully developed
ultimates = pd.DataFrame(
    {
        method: fit.ultimate_.to_frame().values.ravel()
        for method, fit in fits_7_7.items()
    },
    index=range(1, 7),
)

table_7_8 = pd.concat([
    reserves.iloc[1:].sum().to_frame("all periods").T,
    reserves.iloc[1:],
])
table_7_9 = pd.concat([ultimates.sum().to_frame("all periods").T, ultimates])
table_7_8.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,10600143.0,12714477.0,11219478.0,11241879.0,11378317.0
2,27228.0,28101.0,28067.0,28098.0,27664.0
3,586303.0,636809.0,632085.0,633741.0,611161.0
4,918019.0,860619.0,867892.0,866079.0,890036.0
5,2315070.0,1620276.0,1805379.0,1786943.0,1991456.0
6,6753523.0,9568672.0,7886055.0,7927018.0,7858000.0


In [18]:
# Printed values, Table 7.8 (all periods, then origins 2 to 6).
printed_7_8 = {
    "collective": [10_600_143, 27_228, 586_303, 918_019, 2_315_070, 6_753_523],
    "individual": [12_714_477, 28_101, 636_809, 860_619, 1_620_276, 9_568_672],
    "neuhaus": [11_219_478, 28_067, 632_085, 867_892, 1_805_379, 7_886_055],
    "benktander": [11_241_879, 28_098, 633_741, 866_079, 1_786_943, 7_927_018],
    "optimal": [11_378_317, 27_664, 611_161, 890_036, 1_991_456, 7_858_000],
}
for method, printed in printed_7_8.items():
    close(table_7_8[method].values, printed, 1)

Table 7.8 reproduces in full, to the paper's rounding.

In [19]:
table_7_9.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,56940469.0,59054803.0,57559804.0,57582205.0,57718643.0
1,7397862.0,7397862.0,7397862.0,7397862.0,7397862.0
2,8963172.0,8964045.0,8964011.0,8964042.0,8963608.0
3,10431787.0,10482293.0,10477569.0,10479225.0,10456645.0
4,9104890.0,9047490.0,9054763.0,9052950.0,9076907.0
5,7449305.0,6754511.0,6939614.0,6921178.0,7125691.0
6,13593453.0,16408602.0,14725985.0,14766948.0,14697930.0


In [20]:
# Printed values, Table 7.9 (all periods, then origins 1 to 6).
printed_7_9 = {
    "collective": [
        56_940_469,
        7_397_862,
        8_963_172,
        10_431_787,
        9_104_890,
        7_449_305,
        13_593_453,
    ],
    "individual": [
        59_054_803,
        7_397_862,
        8_964_045,
        10_482_293,
        9_047_490,
        6_754_511,
        16_408_602,
    ],
    "neuhaus": [
        57_559_804,
        7_397_862,
        8_964_011,
        10_477_569,
        9_054_763,
        6_939_614,
        14_725_985,
    ],
    "benktander": [
        57_582_205,
        7_397_862,
        8_964_042,
        10_479_225,
        9_052_950,
        6_921_178,
        14_766_948,
    ],
    "optimal": [
        57_718_643,
        7_397_862,
        8_963_608,
        10_456_645,
        9_076_907,
        7_125_691,
        14_697_930,
    ],
}
for method, printed in printed_7_9.items():
    close(table_7_9[method].values, printed, 1)

Table 7.9 reproduces in full, to the paper's rounding.

## Table 7.10: mean squared errors

As for Table 7.5, with the second example.

In [21]:
mse = pd.DataFrame(
    {method: fit.mse_.to_frame().values.ravel() for method, fit in fits_7_7.items()},
    index=range(1, 7),
).iloc[1:]
table_7_10 = mse.div(mse["optimal"], axis=0)
table_7_10.round(6)

,collective,individual,neuhaus,benktander,optimal
2,1.001566,1.001571,1.001342,1.001551,1.0
3,1.029900,1.031833,1.021187,1.024673,1.0
4,1.046373,1.051248,1.029039,1.033989,1.0
5,1.111729,1.146989,1.036940,1.044623,1.0
6,1.228789,1.548852,1.000148,1.000893,1.0


In [22]:
# Printed values, Table 7.10, as corrected by the CAS Exam 7 Syllabus Committee's errata of
# January 2020. Every method's column except the optimal one changed.
printed_7_10 = {
    "collective": [1.001566, 1.029900, 1.046373, 1.111729, 1.228789],
    "individual": [1.001571, 1.031833, 1.051248, 1.146989, 1.548852],
    "neuhaus": [1.001342, 1.021187, 1.029039, 1.036940, 1.000148],
    "benktander": [1.001551, 1.024673, 1.033989, 1.044623, 1.000893],
    "optimal": [1, 1, 1, 1, 1],
}
for method, printed in printed_7_10.items():
    close(table_7_10[method].values, printed, 5e-6)

Table 7.10 reproduces in full, to the paper's rounding, **once the errata is applied**.

## Table 7.11: the paid triangle (general liability, claims made)

The third example compares the 2004 A.M. Best loss development factors to the loss ratio payout factors of the credible loss ratio method. Unlike Tables 7.1 and 7.6, this triangle is **cumulative**, with ten annual origin and development periods, 1994 to 2003.

In [23]:
rows_7_11 = [
    [
        394645,
        991758,
        1601166,
        2048315,
        2286923,
        2464913,
        2578083,
        2632786,
        2678889,
        2691701,
    ],
    [353192, 969542, 1592009, 1974155, 2272665, 2438559, 2566985, 2671750, 2741844],
    [388218, 1078557, 1646205, 2171862, 2443991, 2709179, 2827232, 2934048],
    [389758, 1082588, 1874868, 2427603, 2867146, 3125564, 3277475],
    [449549, 1312383, 2190150, 2989522, 3671202, 4103170],
    [391147, 1398570, 2506347, 3448315, 4134317],
    [683853, 1720774, 3136015, 4112141],
    [586225, 1962547, 3283757],
    [810359, 2149450],
    [648230],
]
long = pd.DataFrame(
    [
        (1994 + i, 1994 + i + k, v)
        for i, r in enumerate(rows_7_11)
        for k, v in enumerate(r)
    ],
    columns=["origin", "valuation", "paid"],
)
cumulative_7_11 = cl.Triangle(
    long, origin="origin", valuation="valuation", columns="paid", cumulative=True
)
table_7_11 = cumulative_7_11.to_frame(origin_as_datetime=False)
table_7_11.columns = range(1, 11)
table_7_11

,1,2,3,4,5,6,7,8,9,10
1994,394645.0,991758.0,1601166.0,2048315.0,2286923.0,2464913.0,2578083.0,2632786.0,2678889.0,2691701.0
1995,353192.0,969542.0,1592009.0,1974155.0,2272665.0,2438559.0,2566985.0,2671750.0,2741844.0,NaN
1996,388218.0,1078557.0,1646205.0,2171862.0,2443991.0,2709179.0,2827232.0,2934048.0,NaN,NaN
1997,389758.0,1082588.0,1874868.0,2427603.0,2867146.0,3125564.0,3277475.0,NaN,NaN,NaN
1998,449549.0,1312383.0,2190150.0,2989522.0,3671202.0,4103170.0,NaN,NaN,NaN,NaN
1999,391147.0,1398570.0,2506347.0,3448315.0,4134317.0,NaN,NaN,NaN,NaN,NaN
2000,683853.0,1720774.0,3136015.0,4112141.0,NaN,NaN,NaN,NaN,NaN,NaN
2001,586225.0,1962547.0,3283757.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002,810359.0,2149450.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003,648230.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Printed values, Table 7.11.
printed_7_11 = np.full((10, 10), np.nan)
for i, r in enumerate(rows_7_11):
    printed_7_11[i, : len(r)] = r
assert np.array_equal(table_7_11.values, printed_7_11, equal_nan=True)

## Table 7.12: inverse of loss ratio payout factors

For each method, the paper compares the inverse of the loss ratio payout factor, $U_i/C_{i,n-i+1}$: the ultimate over the latest paid claims. The A.M. Best column is the published factor, and is an input rather than something to compute.

**The paper does not give the exposures for this triangle**, and the credible methods need them. The loss ratio pattern $\hat m_k$ is claims over exposure, so the factors depend on how exposure varies by origin period, and the Neuhaus weight $p\cdot ELR$ also depends on its level. A constant exposure does not reproduce the table: it gives 7.814 for the last individual factor against the printed 9.285.

So the exposures below were **reconstructed** by least squares: the ten values were chosen so that `HurlimannCLR` reproduces all 50 printed factors (five methods by ten origins), starting from several different levels. They are not the paper's, but one set of ten numbers reproduces all fifty factors to the printed three decimals, which is what a check on the method needs. They are sensitive. Rounding them to the nearest thousand moves some factors by up to 0.0013, so they are kept to the nearest unit.

In [25]:
V_7_12 = [
    2_807_968,
    2_983_227,
    3_193_459,
    3_694_165,
    4_841_427,
    5_358_957,
    6_402_307,
    6_901_179,
    7_904_896,
    6_794_516,
]
premium_7_12 = cl.Triangle(
    pd.DataFrame({"origin": range(1994, 2004), "valuation": 2003, "premium": V_7_12}),
    origin="origin",
    valuation="valuation",
    columns="premium",
    cumulative=True,
)

latest = cumulative_7_11.latest_diagonal.to_frame().values.ravel()
published_7_12 = {
    "A.M. Best": [1.000, 1.010, 1.030, 1.066, 1.114, 1.226, 1.471, 1.986, 3.475, 9.903]
}
methods_7_12 = ["optimal", "benktander", "neuhaus", "collective", "individual"]
table_7_12 = pd.DataFrame(
    {
        **published_7_12,
        **{
            method: cl
            .HurlimannCLR(method=method)
            .fit(cumulative_7_11, sample_weight=premium_7_12)
            .ultimate_.to_frame()
            .values.ravel()
            / latest
            for method in methods_7_12
        },
    },
    index=range(1994, 2004),
)
table_7_12.round(3)

,A.M. Best,optimal,benktander,neuhaus,collective,individual
1994,1.000,1.000,1.000,1.000,1.000,1.000
1995,1.010,1.005,1.005,1.005,1.005,1.005
1996,1.030,1.027,1.027,1.027,1.027,1.027
1997,1.066,1.062,1.062,1.062,1.061,1.062
1998,1.114,1.112,1.113,1.113,1.112,1.113
1999,1.226,1.220,1.221,1.221,1.219,1.222
2000,1.471,1.439,1.439,1.439,1.441,1.438
2001,1.986,1.917,1.914,1.915,1.927,1.903
2002,3.475,3.322,3.328,3.331,3.366,3.245
2003,9.903,9.595,9.652,9.655,9.696,9.285


In [26]:
# Printed values, Table 7.12.
printed_7_12 = {
    "optimal": [1.000, 1.005, 1.027, 1.062, 1.112, 1.220, 1.439, 1.917, 3.322, 9.595],
    "benktander": [
        1.000,
        1.005,
        1.027,
        1.062,
        1.113,
        1.221,
        1.439,
        1.914,
        3.328,
        9.652,
    ],
    "neuhaus": [1.000, 1.005, 1.027, 1.062, 1.113, 1.221, 1.439, 1.915, 3.331, 9.655],
    "collective": [
        1.000,
        1.005,
        1.027,
        1.061,
        1.112,
        1.219,
        1.441,
        1.927,
        3.366,
        9.696,
    ],
    "individual": [
        1.000,
        1.005,
        1.027,
        1.062,
        1.113,
        1.222,
        1.438,
        1.903,
        3.245,
        9.285,
    ],
}
for method, printed in printed_7_12.items():
    close(table_7_12[method].values, printed, 5e-4)

# The paper's conclusion: the A.M. Best factors slightly but systematically overestimate the
# optimal and nearly optimal Benktander and Neuhaus factors.
for method in ("optimal", "benktander", "neuhaus"):
    assert (table_7_12["A.M. Best"].values[1:] > table_7_12[method].values[1:]).all()

Table 7.12 reproduces to the printed three decimals, **given the reconstructed exposures**, and the A.M. Best factors exceed the optimal, Benktander and Neuhaus factors for every origin after the first, as the paper concludes.

This is a weaker check than the others. The ten exposures were fitted to the table, so they absorb the individual column, which is fully determined by them. The collective, Neuhaus, Benktander and optimal columns still have to agree with it independently, and they do (40 values, from the same ten exposures), but it does not establish the paper's own exposures. If they can be recovered from Mack (1997) or the A.M. Best table, this should be rerun with them.